# Direct loss estimation for regression

`DirectLossEstimator` learns an observation-level loss from labeled reference predictions. `DirectLossAnalyzer` fits one loss estimator per ID and estimates performance for a current batch **without current targets**. This notebook uses MAE; the same API supports MSE and RMSE.


In [1]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd

from tinyshift.performance import DirectLossAnalyzer, DirectLossEstimator

rng = np.random.default_rng(42)

## 1. Build labeled reference and unlabeled current batches

The monitored model's prediction is `y_pred`. In this synthetic example, its absolute error grows with `risk`. The current batch for `store_A` has more high-risk observations; `store_B` has more low-risk observations. The `source` column is metadata, so we leave it out of `feature_cols`.


In [2]:
def make_batch(store, risk, *, labeled, source):
    y_pred = 10.0 + 0.5 * risk
    frame = pd.DataFrame({
        "unique_id": store,
        "risk": risk,
        "y_pred": y_pred,
        "source": source,
    })
    if labeled:
        expected_error = 0.3 + 1.5 * risk if store == "store_A" else 0.6 + 0.5 * risk
        frame["y"] = y_pred + expected_error + rng.normal(0, 0.04, len(risk))
    return frame


reference = pd.concat([
    make_batch("store_A", rng.uniform(0, 1, 400), labeled=True, source="reference"),
    make_batch("store_B", rng.uniform(0, 1, 400), labeled=True, source="reference"),
], ignore_index=True)

current = pd.concat([
    make_batch("store_A", rng.beta(8, 2, 120), labeled=False, source="current"),
    make_batch("store_B", rng.beta(2, 8, 120), labeled=False, source="current"),
], ignore_index=True)

print("Reference columns:", reference.columns.tolist())
print("Current columns:", current.columns.tolist())

Reference columns: ['unique_id', 'risk', 'y_pred', 'source', 'y']
Current columns: ['unique_id', 'risk', 'y_pred', 'source']


## 2. Fit one loss model per ID

Within each ID, the analyzer fits the loss model on the first 75% of reference rows and uses the final 25% as a held-out baseline. Keep each ID in the intended reference order; for time-series monitoring, sort by time before fitting. The estimator receives `risk` and the monitored model's `y_pred` internally.


In [3]:
analyzer = DirectLossAnalyzer(
    DirectLossEstimator(metric="mae"),
    validation_fraction=0.25,
).fit(reference, feature_cols=["risk"])

result = analyzer.predict(current)
result

,unique_id,metric,reference_estimated,reference_realized,reference_size,current_estimated,estimated_delta,degradation,current_size
0,store_A,mae,1.085171,1.089046,100,1.505220,0.420049,True,120
1,store_B,mae,0.857237,0.840542,100,0.708394,-0.148843,False,120


## 3. Interpret the estimate

`reference_realized` is the observed MAE on held-out labeled reference rows. `reference_estimated` and `current_estimated` come from the same fitted loss model. `estimated_delta` compares those two estimates; `degradation` is simply whether that difference is positive.

The current frame has no `y`, so the example cannot verify its actual MAE. The result has **no p-value or hypothesis test**. An inaccurate loss model, especially after a large change in the population, can give an inaccurate performance estimate. Compare estimated and realized metrics when current labels eventually arrive.


In [4]:
print(result[[
    "unique_id",
    "reference_realized",
    "reference_estimated",
    "current_estimated",
    "estimated_delta",
    "degradation",
]].to_string(index=False))

unique_id  reference_realized  reference_estimated  current_estimated  estimated_delta  degradation
  store_A            1.089046             1.085171           1.505220         0.420049         True
  store_B            0.840542             0.857237           0.708394        -0.148843        False
